# Case Study 01 — Credit Scoring: Exploratory Data Analysis

**Erick Condoy** · Economist (UNL) · Quant Researcher  
Repository: [credit-risk-lab](https://github.com/EryckFCS/credit-risk-lab)

---

## Objective

Rigorous EDA on the **UCI Default of Credit Card Clients** dataset (N=30,000, Taiwan, 2005).

| Section | Content |
|---------|---------|
| 1 | Data Ingestion |
| 2 | Data Quality Assessment |
| 3 | Target Variable: Class Imbalance |
| 4 | Demographic Variables |
| 5 | Credit Limit & Age |
| 6 | Payment History — Key Risk Driver |
| 7 | Correlation Matrix |
| 8 | Bill vs Payment Amounts |
| 9 | **Information Value (IV) Ranking** |
| 10 | EDA Summary & Modeling Hypotheses |

---

## Dataset Reference

| Attribute | Value |
|-----------|-------|
| Source | UCI ML Repository — ID 350 |
| Observations | 30,000 clients |
| Features | 23 predictor variables |
| Target | `DEFAULT` (1=default, 0=no default) |
| Period | April–September 2005 |
| Geography | Taiwan (private bank) |

**Download:** `bash data/download.sh` → genera `data/credit_card_default.parquet`  
**Citation:** Yeh, I.C. & Lien, C. (2009). *The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients.* Expert Systems with Applications, 36(2), 2473–2480.

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)

REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'axes.edgecolor': '#dcd9d5',
    'axes.labelcolor': '#28251d',
    'xtick.color': '#7a7974',
    'ytick.color': '#7a7974',
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'semibold',
    'figure.dpi': 120,
})

TEAL   = '#01696f'
MAROON = '#a12c7b'
GRAY   = '#bab9b4'
GREEN  = '#437a22'
ORANGE = '#964219'

print(f'Python {sys.version}')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

## 1. Data Ingestion

In [ ]:
DATA_DIR = Path('../data')

# Priority: .parquet (from download.sh) → .csv → .xls/.xlsx (manual download)
def _find_dataset(data_dir: Path) -> Path:
    for pattern in ('*.parquet', '*.csv', '*.xlsx', '*.xls'):
        candidates = list(data_dir.glob(pattern))
        if candidates:
            return candidates[0]
    raise FileNotFoundError(
        'Dataset not found. Run:\n'
        '    bash data/download.sh\n'
        'Or download manually from: https://archive.ics.uci.edu/dataset/350'
    )

fpath = _find_dataset(DATA_DIR)
print(f'Loading: {fpath.name}  ({fpath.stat().st_size / 1024:.0f} KB)')

if fpath.suffix == '.parquet':
    raw = pd.read_parquet(fpath)
    # download.sh already normalises column names to UPPER
    raw.columns = raw.columns.str.strip().str.upper().str.replace(' ', '_').str.replace('.', '_')
elif fpath.suffix in ('.xls', '.xlsx'):
    # Original UCI Excel file has a description row (row 0); data starts at row 1
    raw = pd.read_excel(fpath, header=1)
    raw.columns = raw.columns.str.strip().str.upper().str.replace(' ', '_').str.replace('.', '_')
else:
    raw = pd.read_csv(fpath)
    raw.columns = raw.columns.str.strip().str.upper().str.replace(' ', '_').str.replace('.', '_')

print(f'Shape: {raw.shape}')
raw.head(3)

In [ ]:
df = raw.copy()

# Normalise column names to lowercase for ergonomic access
df.columns = df.columns.str.lower()

# Resolve target column — handles both 'default' and 'default_payment_next_month'
target_candidates = [c for c in df.columns if 'default' in c]
if not target_candidates:
    raise KeyError(f'No default column found. Available: {list(df.columns)}')
target_raw = target_candidates[0]

# Rename id col if present
id_col_candidates = [c for c in df.columns if c == 'id']
rename_map = {target_raw: 'default'}
if id_col_candidates:
    rename_map['id'] = 'client_id'

df = df.rename(columns=rename_map)
TARGET = 'default'
ID_COL = 'client_id' if 'client_id' in df.columns else None
FEATURES = [c for c in df.columns if c not in ([TARGET] + ([ID_COL] if ID_COL else []))]

print('Target column :', TARGET)
print('ID column     :', ID_COL)
print('Feature count :', len(FEATURES))
print('Features      :', FEATURES)

## 2. Data Quality Assessment

In [ ]:
missing = pd.DataFrame({
    'n_missing': df.isnull().sum(),
    'pct_missing': df.isnull().mean() * 100,
    'dtype': df.dtypes
}).sort_values('n_missing', ascending=False)

print('=== Missing Value Report ===')
print(missing[missing.n_missing > 0].to_string() if missing.n_missing.sum() > 0
      else '✓ No missing values detected.')

n_dupes = df.duplicated(subset=FEATURES).sum()
print(f'\nDuplicate rows (on features): {n_dupes} ({n_dupes/len(df)*100:.2f}%)')

In [ ]:
# Known encoding issues in UCI dataset — undocumented codes 0 and 5/6 in EDUCATION/MARRIAGE
for col, valid, label in [
    ('sex',       [1, 2],       'SEX'),
    ('education', [1, 2, 3, 4], 'EDUCATION'),
    ('marriage',  [1, 2, 3],    'MARRIAGE'),
]:
    if col in df.columns:
        invalid = df[~df[col].isin(valid)][col].value_counts()
        if len(invalid):
            print(f'[WARN] {label} — undocumented codes: {invalid.to_dict()}')
            print(f'       → These rows will be kept and mapped to "Others" in feature engineering.')
        else:
            print(f'[OK]  {label} — all values in valid set')

## 3. Target Variable: Class Imbalance

In [ ]:
target_counts = df[TARGET].value_counts()
default_rate  = df[TARGET].mean()

print(f'Default rate   : {default_rate:.2%}')
print(f'Good (0)       : {target_counts[0]:,}')
print(f'Bad  (1)       : {target_counts[1]:,}')
print(f'Good:Bad ratio : {target_counts[0]/target_counts[1]:.1f}:1')
print(f'\nNote: moderate imbalance — SMOTE or class_weight="balanced" recommended in modeling.')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
bars = axes[0].bar(['Good (0)', 'Bad (1)'], target_counts.values,
                   color=[GREEN, MAROON], width=0.5, edgecolor='none')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, target_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}\n({val/len(df):.1%})', ha='center', va='bottom', fontsize=9)
wedges, _, autotexts = axes[1].pie(
    target_counts.values, labels=['Good', 'Bad'], colors=[GREEN, MAROON],
    autopct='%1.1f%%', startangle=90, wedgeprops=dict(width=0.55))
axes[1].set_title('Default Rate (Donut)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '01_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Demographic Variables

In [ ]:
cat_map = {
    'sex':       {1: 'Male', 2: 'Female'},
    'education': {1: 'Grad School', 2: 'University', 3: 'High School', 4: 'Others'},
    'marriage':  {1: 'Married', 2: 'Single', 3: 'Others'},
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (col, mapping) in zip(axes, cat_map.items()):
    tmp = df[df[col].isin(mapping.keys())].copy()
    tmp['label'] = tmp[col].map(mapping)
    dr = tmp.groupby('label')[TARGET].mean().sort_values(ascending=False)
    bars = ax.bar(dr.index, dr.values, color=TEAL, edgecolor='none', width=0.5)
    ax.axhline(default_rate, color=MAROON, linestyle='--', lw=1.2, label=f'Overall {default_rate:.1%}')
    ax.set_title(f'Default Rate by {col.upper()}')
    ax.set_ylabel('Default Rate')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=8)
    for bar, val in zip(bars, dr.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8)
    ax.tick_params(axis='x', labelrotation=15)
plt.tight_layout()
plt.savefig(REPORTS_DIR / '02_default_rate_demographics.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Credit Limit & Age Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(df['limit_bal'] / 1000, bins=50, color=TEAL, edgecolor='none', alpha=0.8)
axes[0, 0].set_title('Credit Limit Distribution')
axes[0, 0].set_xlabel('Credit Limit (NT$ thousands)')
for label, grp in df.groupby(TARGET):
    axes[0, 1].hist(grp['limit_bal'] / 1000, bins=50, alpha=0.55,
                   color=GREEN if label==0 else MAROON,
                   label='Good' if label==0 else 'Bad',
                   edgecolor='none', density=True)
axes[0, 1].set_title('Credit Limit by Default Status')
axes[0, 1].legend(fontsize=8)
axes[1, 0].hist(df['age'], bins=40, color=ORANGE, edgecolor='none', alpha=0.8)
axes[1, 0].set_title('Age Distribution')
axes[1, 0].set_xlabel('Age (years)')
df['age_decile'] = pd.qcut(df['age'], q=10, labels=False, duplicates='drop') + 1
age_dr = df.groupby('age_decile').agg(default_rate=(TARGET, 'mean'), age_mid=('age', 'median')).reset_index()
axes[1, 1].plot(age_dr['age_mid'], age_dr['default_rate'], marker='o', color=TEAL, lw=2, markersize=5)
axes[1, 1].axhline(default_rate, color=MAROON, linestyle='--', lw=1.2, label='Overall rate')
axes[1, 1].set_title('Default Rate by Age Decile')
axes[1, 1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1, 1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(REPORTS_DIR / '03_credit_limit_age.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Payment History (PAY_0–PAY_6) — Key Risk Driver

In [ ]:
pay_cols = sorted(
    [c for c in df.columns if c.startswith('pay_') and c[4:].isdigit()],
    key=lambda x: int(x.split('_')[1])
)[:6]

if not pay_cols:
    # Fallback for datasets where pay_0 is the first col name
    pay_cols = sorted([c for c in df.columns if c in
                       ['pay_0','pay_2','pay_3','pay_4','pay_5','pay_6']])

pay_labels = {-2:'No credit',-1:'Paid duly',0:'Min paid',1:'1m delay',
              2:'2m delay',3:'3m+',4:'4m+',5:'5m+',6:'6m+',7:'7m+',8:'8m+'}
print(f'Payment status columns identified: {pay_cols}')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, col in zip(axes, pay_cols):
    dr = df.groupby(col)[TARGET].mean().reset_index()
    dr['label'] = dr[col].map(pay_labels).fillna(dr[col].astype(str))
    colors = [MAROON if v >= 1 else TEAL for v in dr[col]]
    ax.bar(dr['label'], dr[TARGET], color=colors, edgecolor='none', width=0.6)
    ax.axhline(default_rate, color=GRAY, linestyle='--', lw=1.2)
    ax.set_title(f'{col.upper()} — Default Rate')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.tick_params(axis='x', labelrotation=30, labelsize=7)
plt.tight_layout()
plt.savefig(REPORTS_DIR / '04_payment_history_default_rate.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Correlation Matrix

In [ ]:
numeric_feats = df[FEATURES + [TARGET]].select_dtypes(include=np.number).columns.tolist()
corr = df[numeric_feats].corr()
target_corr = corr[TARGET].drop(TARGET).sort_values(ascending=False)
top_feats = target_corr.abs().nlargest(14).index.tolist() + [TARGET]
corr_sub = df[top_feats].corr()
mask = np.triu(np.ones_like(corr_sub, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(corr_sub, mask=mask, cmap=sns.diverging_palette(220, 20, as_cmap=True),
            center=0, vmin=-1, vmax=1, annot=True, fmt='.2f',
            annot_kws={'size': 7}, linewidths=0.3, ax=axes[0])
axes[0].set_title('Correlation Matrix (Top 14 Features)')
colors_bar = [MAROON if v > 0 else TEAL for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors_bar, edgecolor='none')
axes[1].axvline(0, color='#dcd9d5', lw=1)
axes[1].set_title('Correlation with Default Target')
axes[1].set_xlabel('Pearson r')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '05_correlation_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Bill Amounts vs. Payment Amounts

In [ ]:
bill_cols    = sorted([c for c in df.columns if c.startswith('bill_amt')])
pay_amt_cols = sorted([c for c in df.columns if c.startswith('pay_amt')])

df['avg_bill']    = df[bill_cols].mean(axis=1)
df['avg_pay_amt'] = df[pay_amt_cols].mean(axis=1)
df['utilization'] = np.clip(df['avg_bill'] / df['limit_bal'].replace(0, np.nan), 0, 5)
df['pay_ratio']   = np.clip(df['avg_pay_amt'] / df['avg_bill'].replace(0, np.nan), 0, 5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, grp in df.groupby(TARGET):
    c = GREEN if label==0 else MAROON
    n = 'Good' if label==0 else 'Bad'
    axes[0].hist(grp['utilization'].clip(0,2), bins=50, alpha=0.55, color=c, label=n, density=True, edgecolor='none')
    axes[1].hist(grp['pay_ratio'].clip(0,3),   bins=50, alpha=0.55, color=c, label=n, density=True, edgecolor='none')
axes[0].set_title('Credit Utilization by Default Status')
axes[0].set_xlabel('Utilization Ratio (avg_bill / limit)')
axes[0].legend(fontsize=8)
axes[1].set_title('Payment Ratio by Default Status')
axes[1].set_xlabel('Payment Ratio (avg_pay / avg_bill)')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(REPORTS_DIR / '06_utilization_payment_ratio.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. Information Value (IV) Ranking

IV mide la capacidad predictiva de cada variable respecto al target.  
Umbral industria (Siddiqi, 2006): `IV < 0.02` Useless · `0.02–0.10` Weak · `0.10–0.30` Medium · `0.30–0.50` Strong · `>0.50` Very Strong.

In [ ]:
def compute_iv(series: pd.Series, target: pd.Series, bins: int = 10) -> float:
    """Compute Information Value (WoE method) for a single feature."""
    df_iv = pd.DataFrame({'x': series, 'y': target})
    if series.dtype == 'object' or series.nunique() <= bins:
        df_iv['bin'] = series.astype(str)
    else:
        df_iv['bin'] = pd.qcut(series, q=bins, duplicates='drop').astype(str)
    total_good = max((target == 0).sum(), 1)
    total_bad  = max((target == 1).sum(), 1)
    grouped = df_iv.groupby('bin')['y'].agg(
        bad=lambda x:  (x == 1).sum(),
        good=lambda x: (x == 0).sum()
    )
    grouped['dist_bad']  = grouped['bad']  / total_bad
    grouped['dist_good'] = grouped['good'] / total_good
    grouped = grouped[(grouped['dist_bad'] > 0) & (grouped['dist_good'] > 0)]
    grouped['woe'] = np.log(grouped['dist_bad'] / grouped['dist_good'])
    grouped['iv']  = (grouped['dist_bad'] - grouped['dist_good']) * grouped['woe']
    return grouped['iv'].sum()


iv_results = {
    col: compute_iv(df[col], df[TARGET])
    for col in FEATURES
    if col in df.columns
}

iv_df = (
    pd.DataFrame.from_dict(iv_results, orient='index', columns=['IV'])
    .sort_values('IV', ascending=False)
    .assign(strength=lambda x: x['IV'].map(lambda v:
        'Useless'     if v < 0.02 else
        'Weak'        if v < 0.10 else
        'Medium'      if v < 0.30 else
        'Strong'      if v < 0.50 else 'Very Strong'))
)

print(f'Features with IV > 0.10 (Medium+): {(iv_df["IV"] > 0.10).sum()}')
print(f'Top predictor: {iv_df.index[0]} (IV={iv_df["IV"].iloc[0]:.4f})')
display(iv_df.style.format({'IV': '{:.4f}'}).bar(subset=['IV'], color=TEAL))

In [ ]:
strength_colors = {
    'Very Strong': '#01696f', 'Strong': '#4f98a3',
    'Medium': '#d19900', 'Weak': '#da7101', 'Useless': '#dcd9d5'
}
bar_colors = iv_df['strength'].map(strength_colors).values

fig, ax = plt.subplots(figsize=(9, max(5, len(iv_df) * 0.35)))
ax.barh(iv_df.index[::-1], iv_df['IV'][::-1], color=bar_colors[::-1], edgecolor='none')
ax.axvline(0.02, color='#bab9b4', linestyle=':',  lw=1.2, label='Useless (0.02)')
ax.axvline(0.10, color=ORANGE,    linestyle='--', lw=1.2, label='Weak→Medium (0.10)')
ax.axvline(0.30, color=TEAL,      linestyle='--', lw=1.2, label='Medium→Strong (0.30)')
ax.set_xlabel('Information Value (IV)')
ax.set_title('Feature Predictive Power — IV Ranking (WoE method)', fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '07_iv_ranking.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. EDA Summary & Modeling Hypotheses

In [ ]:
summary = {
    'n_observations':           len(df),
    'n_features':               len(FEATURES),
    'default_rate':             f'{default_rate:.2%}',
    'class_imbalance_ratio':    f'{target_counts[0]/target_counts[1]:.1f}:1',
    'avg_credit_limit_NT$':     f'{df.limit_bal.mean():,.0f}',
    'avg_age':                  f'{df.age.mean():.1f} years',
    'missing_values':           df.isnull().sum().sum(),
    'features_IV_medium_plus':  int((iv_df['IV'] > 0.10).sum()),
    'top_predictor':            f'{iv_df.index[0]} (IV={iv_df["IV"].iloc[0]:.4f})',
    'engineered_features':      'utilization, pay_ratio, age_decile',
}

print('=== EDA Summary ===')
for k, v in summary.items():
    print(f'  {k:<35} {v}')

print()
print('=== Modeling Hypotheses (to be validated in 02_feature_engineering.ipynb) ===')
hypotheses = [
    'H1: PAY_0 (most recent payment status) is the strongest single predictor (highest IV).',
    'H2: Higher credit utilization increases default probability (non-linear effect).',
    'H3: Clients with graduate education show lower default rates than high-school level.',
    'H4: Low pay_ratio (avg_pay / avg_bill) is a strong delinquency signal.',
    'H5: Age has a U-shaped relationship with default — younger clients are riskier.',
    'H6: Female clients default at a lower rate than males (requires chi-square test).',
]
for h in hypotheses:
    print(f'  {h}')

# Export summary to CSV for report
pd.Series(summary).to_csv(REPORTS_DIR / 'eda_summary.csv', header=['value'])
print()
print('Figures saved to   :', REPORTS_DIR.resolve())
print('Next notebook      : 02_feature_engineering.ipynb')